In [ ]:
# Main 2xT4 Stream1 piece-transformer benchmark config
GITHUB_REPO_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"
GITHUB_REF = "stream1-transformer-rtx3070-final-18e5996"
GITHUB_EXPECTED_COMMIT = "18e5996"
KAGGLE_MODEL_SOURCE = "vladkuznetsov266/megaminx-qtransformer-1782210824/PyTorch/default/1"
MODEL_SOURCE_SLUG = "megaminx-qtransformer-1782210824"
MODEL_INPUT_ROOTS = [
    "/kaggle/input/megaminx-qtransformer-1782210824/PyTorch/default/1",
    "/kaggle/input/megaminx-qtransformer-1782210824/pytorch/default/1",
]
WEIGHT_OUT_DIR = "/kaggle/working/stream1_transformer_weights_fp16"

BEAM_WIDTH = 1_048_576
START_PUZZLE_ID = 0
PUZZLE_COUNT = 1
DEPTH_LIMIT = 3
RUN_TIMEOUT_SEC = 0
CUDA_ARCHITECTURES = "75"

TORCHRUN_NNODES = 1
TORCHRUN_NPROC_PER_NODE = 2
TORCHRUN_NODE_RANK = 0
TORCHRUN_RDZV_BACKEND = "c10d"
TORCHRUN_RDZV_ENDPOINT = "127.0.0.1:29500"
TORCHRUN_RDZV_ID = "stream1_transformer_smoke"

ENABLE_DEBUG = False
ENABLE_DEPTH_LOGS = False
ENABLE_DEBUG_LOGS = False
DEBUG_STREAM_TIMING = False
DEBUG_INFERENCE_TRACE = False
DEBUG_PATH_TRACE = False
DEBUG_FINAL_VALIDATE = False
DEBUG_FINAL_EXCHANGE_TRACE = False
DEBUG_FINAL_HISTOGRAM_TRACE = False
DEBUG_STREAM4_HISTOGRAM_TRACE = False
DEBUG_DEPTH_FLOW_TRACE = False
DEBUG_PIPELINE_STATS = False

RUNTIME_CONFIG_MODE = "manual"
SHARD_BUFFER_COUNT = 2
STREAM4_BATCH_ALIGNMENT = 1024
SHARD_CAPACITY_SCALE_PPM = 1050000
GLOBAL_SPILL_CAPACITY = 0
STREAM5_RECV_CAPACITY_SCALE_PPM = 1000000
GPU_HEADROOM_BYTES = 256 * 1024**2
B_MICRO = 512
STREAM1_CONCURRENCY = 1
STREAM3_RING_SLOTS = 2
SHARD_COUNT = 4
STREAM4_ACTIVE_SORT_SLOTS = 2
STREAM4_BATCH_CANDIDATES = 32_768
STREAM4_TRIGGER_CANDIDATES = 65_536

DEPTH_LOG_EVERY = 1
PUZZLE_LOG_EVERY = 1
HISTORY_MODE = "ram"
HISTORY_SLOT_COUNT = 2
HISTORY_WORKERS = 1
HISTORY_RAM_BYTES = 4 * 1024**3
HISTORY_DISK_BYTES = 0
HISTORY_DISK_PATH = "/tmp/beam_history_transformer_smoke"
SOLVED_NEIGHBORHOOD_RADIUS = 3
SOLVED_NEIGHBORHOOD_MAX_ENTRIES = 0
STREAM2_SUFFIX_RADIUS = 0
STREAM2_SUFFIX_BACKEND = "composed_permutations"
STREAM2_SUFFIX_MAX_COUNT = 0
STOP_ON_FAILURE = True
LIVE_LOG_RANKS = [0]

# Benchmark-only controls; solver config above is kept for shared preflight sanity checks.
BENCH_PUZZLE_ID = 0
BENCH_GPUS = [0, 1]
BENCH_REPORT_DIR = "/kaggle/working/stream1_transformer_benchmark_reports"
BENCH_LOG_DIR = "/kaggle/working/stream1_transformer_benchmark_logs"


SHAPE_REPEATS = 2
BENCH_SHAPES = [
    (256, 1), (384, 1), (512, 1), (768, 1), (1024, 1),
    (256, 2), (384, 2), (512, 2),
]
POLICY_REPEATS = 5
POLICY_CONFIGS = [
    ("full_final", {
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ONLY": "0",
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ATTENTION": "0",
    }),
    ("cls_full_attention", {
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ONLY": "1",
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ATTENTION": "0",
    }),
    ("cls_q1_q64", {
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ONLY": "1",
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ATTENTION": "1",
        "BEAM_STREAM1_TRANSFORMER_CLS_ATTENTION_POLICY": "cutlass",
    }),
    ("cls_q1_q32", {
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ONLY": "1",
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ATTENTION": "1",
        "BEAM_STREAM1_TRANSFORMER_CLS_ATTENTION_POLICY": "q32k64",
    }),
    ("full_q32k64", {
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ONLY": "1",
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ATTENTION": "0",
        "BEAM_STREAM1_TRANSFORMER_ATTENTION_TILE_POLICY": "q32k64",
    }),
    ("full_exact32", {
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ONLY": "1",
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ATTENTION": "0",
        "BEAM_STREAM1_TRANSFORMER_ATTENTION_MAX_K_POLICY": "exact32",
    }),
    ("full_q64k64v4", {
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ONLY": "1",
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ATTENTION": "0",
        "BEAM_STREAM1_TRANSFORMER_ATTENTION_TILE_POLICY": "q64k64v4",
    }),
    ("ln_persistent", {
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ONLY": "1",
        "BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ATTENTION": "0",
        "BEAM_STREAM1_TRANSFORMER_LAYERNORM_ROWS_POLICY": "persistent",
    }),
]


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

WORK_DIR = Path('/kaggle/working')
TMP_DIR = Path('/tmp')
REPO_DIR = TMP_DIR / 'beam_solver_transformer_benchmark'
CUTLASS_DIR = TMP_DIR / 'cutlass'
BUILD_DIR = TMP_DIR / 'beam_build_transformer_benchmark'
WEIGHT_OUT_DIR = Path(WEIGHT_OUT_DIR)
RUN_LOG_DIR = WORK_DIR / 'stream1_transformer_benchmark_logs'
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)


def run_checked(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print('+ ' + ' '.join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, env=env, check=True)


def run_capture(cmd, cwd=None, env=None, check=True):
    cmd = [str(part) for part in cmd]
    print('+ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout, end='', flush=True)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result


def round_up(value, alignment):
    return ((int(value) + int(alignment) - 1) // int(alignment)) * int(alignment)


def torchrun_world_size():
    world_size = int(TORCHRUN_NNODES) * int(TORCHRUN_NPROC_PER_NODE)
    if world_size <= 0:
        raise ValueError(f'invalid torchrun topology: nnodes={TORCHRUN_NNODES} nproc={TORCHRUN_NPROC_PER_NODE}')
    return world_size


def derived_values():
    world_size = torchrun_world_size()
    beam_alignment = world_size * int(SHARD_COUNT) * int(STREAM4_BATCH_ALIGNMENT)
    global_beam_effective = round_up(BEAM_WIDTH, beam_alignment)
    local_beam_width = global_beam_effective // world_size
    logical_shard_size = (local_beam_width + int(SHARD_COUNT) - 1) // int(SHARD_COUNT)
    shard_capacity = round_up((logical_shard_size * int(SHARD_CAPACITY_SCALE_PPM) + 999999) // 1000000, STREAM4_BATCH_ALIGNMENT)
    stream3_batch = int(STREAM3_RING_SLOTS) * int(B_MICRO) * 24
    return {
        'world_size': world_size,
        'beam_alignment': beam_alignment,
        'global_beam_effective': global_beam_effective,
        'local_beam_width': local_beam_width,
        'logical_shard_size': logical_shard_size,
        'shard_capacity': shard_capacity,
        'stream3_batch': stream3_batch,
    }


def disk_line(path):
    usage = shutil.disk_usage(path)
    return f'{path}: free={usage.free} total={usage.total}'


def preflight():
    values = derived_values()
    print('KAGGLE_MODEL_SOURCE=', KAGGLE_MODEL_SOURCE, flush=True)
    print('GITHUB_REF=', GITHUB_REF, flush=True)
    print('torchrun_topology=', {
        'nnodes': TORCHRUN_NNODES,
        'nproc_per_node': TORCHRUN_NPROC_PER_NODE,
        'node_rank': TORCHRUN_NODE_RANK,
        'world_size': values['world_size'],
        'rdzv_endpoint': TORCHRUN_RDZV_ENDPOINT,
    }, flush=True)
    print('derived_config=', values, flush=True)
    print('disk_tmp=', disk_line('/tmp'), flush=True)
    print('disk_working=', disk_line('/kaggle/working'), flush=True)
    run_capture(['nvidia-smi'], check=False)
    import torch
    gpu_count = torch.cuda.device_count()
    print('torch_cuda_device_count=', gpu_count, flush=True)
    if gpu_count < int(TORCHRUN_NPROC_PER_NODE):
        raise RuntimeError(f'2xT4 smoke requires at least {TORCHRUN_NPROC_PER_NODE} CUDA devices, found {gpu_count}')
    if int(STREAM1_CONCURRENCY) > int(STREAM3_RING_SLOTS):
        raise RuntimeError('STREAM1_CONCURRENCY must be <= STREAM3_RING_SLOTS')
    if values['stream3_batch'] > values['shard_capacity']:
        raise RuntimeError(f'stream3_batch={values["stream3_batch"]} exceeds shard_capacity={values["shard_capacity"]}')
    if int(STREAM4_BATCH_CANDIDATES) > values['shard_capacity']:
        raise RuntimeError(f'STREAM4_BATCH_CANDIDATES={STREAM4_BATCH_CANDIDATES} exceeds shard_capacity={values["shard_capacity"]}')
    if int(STREAM4_TRIGGER_CANDIDATES) > values['shard_capacity']:
        raise RuntimeError(f'STREAM4_TRIGGER_CANDIDATES={STREAM4_TRIGGER_CANDIDATES} exceeds shard_capacity={values["shard_capacity"]}')
    return values


def cleanup_path(path):
    path = Path(path)
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()


def find_model_checkpoint():
    input_root = Path('/kaggle/input')
    all_pth = sorted(input_root.rglob('*.pth')) if input_root.exists() else []
    candidate_roots = [Path(path) for path in MODEL_INPUT_ROOTS if Path(path).exists()]
    if not candidate_roots and input_root.exists():
        candidate_roots = sorted({path for path in input_root.rglob('*') if path.is_dir() and MODEL_SOURCE_SLUG in str(path)})
    matches = []
    for root in candidate_roots:
        matches.extend(root.rglob('*.pth'))
    matches = sorted(set(matches))
    if len(matches) != 1:
        message = [
            'expected exactly one transformer .pth under the configured Kaggle model source',
            f'configured_model_source={KAGGLE_MODEL_SOURCE}',
            'candidate_roots=' + json.dumps([str(path) for path in candidate_roots], indent=2),
            'matched_pth=' + json.dumps([str(path) for path in matches], indent=2),
            'all_discovered_pth=' + json.dumps([str(path) for path in all_pth], indent=2),
        ]
        raise RuntimeError('\n'.join(message))
    return matches[0]


preflight_values = preflight()
for path in (REPO_DIR, BUILD_DIR, WEIGHT_OUT_DIR):
    cleanup_path(path)

run_checked(['git', 'clone', '--branch', GITHUB_REF, '--depth', '1', GITHUB_REPO_URL, REPO_DIR])
commit = run_capture(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR).stdout.strip()
print('GITHUB_COMMIT=', commit, flush=True)
if GITHUB_EXPECTED_COMMIT and not commit.startswith(GITHUB_EXPECTED_COMMIT):
    raise RuntimeError(f'expected GitHub commit {GITHUB_EXPECTED_COMMIT}, got {commit}')
checkpoint_path = find_model_checkpoint()
print('selected_transformer_checkpoint=', checkpoint_path, flush=True)
run_checked([
    sys.executable, REPO_DIR / 'tools' / 'export_stream1.py',
    '--weights', checkpoint_path,
    '--out', WEIGHT_OUT_DIR,
    '--format', 'piece-transformer',
    '--dtype', 'fp16',
    '--num-classes', '120',
], cwd=REPO_DIR)
manifest = json.loads((WEIGHT_OUT_DIR / 'manifest.json').read_text(encoding='utf-8'))
if manifest.get('backend') != 'piece_transformer':
    raise RuntimeError(f'exported manifest backend is not piece_transformer: {manifest.get("backend")!r}')
print('exported_manifest_summary=', {
    'backend': manifest.get('backend'),
    'dtype': manifest.get('dtype'),
    'seq_len': manifest.get('seq_len'),
    'd_model': manifest.get('d_model'),
    'layers': manifest.get('num_layers'),
    'output_dim': manifest.get('output_dim'),
}, flush=True)

if not (CUTLASS_DIR / 'include').exists():
    cleanup_path(CUTLASS_DIR)
    run_checked(['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/cutlass.git', CUTLASS_DIR])

run_checked([
    'cmake', '-S', REPO_DIR, '-B', BUILD_DIR, '-GNinja',
    '-DCMAKE_BUILD_TYPE=Release',
    f'-DBEAM_CUDA_ARCHITECTURES={CUDA_ARCHITECTURES}',
    f'-DCUTLASS_DIR={CUTLASS_DIR}',
    f'-DBEAM_ENABLE_DEBUG={"ON" if ENABLE_DEBUG else "OFF"}',
    f'-DBEAM_ENABLE_DEPTH_LOGS={"ON" if ENABLE_DEPTH_LOGS else "OFF"}',
    f'-DBEAM_ENABLE_DEBUG_LOGS={"ON" if ENABLE_DEBUG_LOGS else "OFF"}',
    f'-DBEAM_DEBUG_STREAM_TIMING={"ON" if DEBUG_STREAM_TIMING else "OFF"}',
    f'-DBEAM_DEBUG_INFERENCE_TRACE={"ON" if DEBUG_INFERENCE_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_PATH_TRACE={"ON" if DEBUG_PATH_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_FINAL_VALIDATE={"ON" if DEBUG_FINAL_VALIDATE else "OFF"}',
    f'-DBEAM_DEBUG_FINAL_EXCHANGE_TRACE={"ON" if DEBUG_FINAL_EXCHANGE_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_FINAL_HISTOGRAM_TRACE={"ON" if DEBUG_FINAL_HISTOGRAM_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_STREAM4_HISTOGRAM_TRACE={"ON" if DEBUG_STREAM4_HISTOGRAM_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_DEPTH_FLOW_TRACE={"ON" if DEBUG_DEPTH_FLOW_TRACE else "OFF"}',
    f'-DBEAM_DEBUG_PIPELINE_STATS={"ON" if DEBUG_PIPELINE_STATS else "OFF"}',
], cwd=REPO_DIR)
run_checked(['cmake', '--build', BUILD_DIR, '--target', 'stream_benchmark', '-j', '2'])
print('stream_benchmark_build_done=1', flush=True)


In [ ]:
import csv
import hashlib
import json
import os
from pathlib import Path
import re
import statistics
import subprocess
import time

BENCH_REPORT_DIR = Path(BENCH_REPORT_DIR)
BENCH_LOG_DIR = Path(BENCH_LOG_DIR)
DUMP_DIR = Path('/kaggle/working/stream1_transformer_score_dumps')
BENCH_REPORT_DIR.mkdir(parents=True, exist_ok=True)
BENCH_LOG_DIR.mkdir(parents=True, exist_ok=True)
DUMP_DIR.mkdir(parents=True, exist_ok=True)

row_pattern = re.compile(
    r'stream1_transformer_micro\s+'
    r'b_micro=(?P<b_micro>\d+)\s+'
    r'concurrency=(?P<concurrency>\d+)\s+'
    r'rows_per_launch_group=(?P<rows>\d+)\s+'
    r'ms_per_launch_group=(?P<ms>[0-9.]+)\s+'
    r'parents_per_sec=(?P<parents>[0-9.]+)\s+'
    r'candidates_per_sec=(?P<candidates>[0-9.]+)\s+'
    r'.*?checksum=(?P<checksum>\d+)\s+'
    r'score_key_digest=(?P<digest>\d+)\s+'
    r'.*?scratch_bytes=(?P<scratch>\d+)'
)


def parse_row(line, gpu, phase, label, repeat):
    match = row_pattern.search(line)
    if not match:
        return None
    values = match.groupdict()
    return {
        'gpu': int(gpu),
        'phase': phase,
        'label': label,
        'repeat': int(repeat),
        'b_micro': int(values['b_micro']),
        'concurrency': int(values['concurrency']),
        'rows_per_launch_group': int(values['rows']),
        'ms_per_launch_group': float(values['ms']),
        'parents_per_sec': float(values['parents']),
        'candidates_per_sec': float(values['candidates']),
        'checksum': int(values['checksum']),
        'score_key_digest': int(values['digest']),
        'scratch_bytes': int(values['scratch']),
    }


def run_one(gpu, phase, label, repeat, b_micro, concurrency, extra_env, write_dump):
    stem = f'{phase}_gpu{gpu}_{label}_r{repeat}_b{b_micro}_c{concurrency}'
    report_path = BENCH_REPORT_DIR / f'{stem}.md'
    log_path = BENCH_LOG_DIR / f'{stem}.log'
    dump_path = DUMP_DIR / f'{stem}.bin'
    env = os.environ.copy()
    env.update({
        'CUDA_VISIBLE_DEVICES': str(gpu),
        'BEAM_WEIGHT_DIR': str(WEIGHT_OUT_DIR),
        'BEAM_STREAM_BENCH_REPORT': str(report_path),
        'BEAM_STREAM1_TRANSFORMER_GRAPH_BENCH': '1',
        'BEAM_STREAM1_TRANSFORMER_BLOCK51': '1',
        'BEAM_STREAM1_TRANSFORMER_B_MICRO': str(b_micro),
        'BEAM_STREAM1_TRANSFORMER_CONCURRENCY': str(concurrency),
    })
    env.update({str(key): str(value) for key, value in extra_env.items()})
    if write_dump:
        if dump_path.exists():
            dump_path.unlink()
        env['BEAM_STREAM1_TRANSFORMER_SCORE_DUMP'] = str(dump_path)
    cmd = [str(BUILD_DIR / 'stream_benchmark'), str(BENCH_PUZZLE_ID)]
    print('RUN_T4_POINT_START', {
        'gpu': gpu, 'phase': phase, 'label': label, 'repeat': repeat,
        'b_micro': b_micro, 'concurrency': concurrency,
    }, flush=True)
    start = time.time()
    parsed = []
    with log_path.open('w', buffering=1, encoding='utf-8') as log:
        proc = subprocess.Popen(
            cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            log.write(line)
            print(line, end='', flush=True)
            row = parse_row(line, gpu, phase, label, repeat)
            if row is not None:
                parsed.append(row)
        rc = proc.wait()
    elapsed = time.time() - start
    print('RUN_T4_POINT_DONE', {
        'gpu': gpu, 'phase': phase, 'label': label, 'repeat': repeat,
        'rc': rc, 'seconds': elapsed, 'rows': len(parsed),
    }, flush=True)
    if rc != 0:
        raise RuntimeError(f'{stem} failed with return code {rc}; log={log_path}')
    if len(parsed) != 1:
        raise RuntimeError(f'{stem} expected one parsed row, got {len(parsed)}; log={log_path}')
    row = parsed[0]
    row['dump_path'] = str(dump_path) if write_dump else ''
    row['dump_sha256'] = hashlib.sha256(dump_path.read_bytes()).hexdigest() if write_dump else ''
    return row


all_rows = []
errors = []

shape_env = {
    'BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ONLY': '1',
    'BEAM_STREAM1_TRANSFORMER_FINAL_CLS_ATTENTION': '0',
}
for gpu in BENCH_GPUS:
    for b_micro, concurrency in BENCH_SHAPES:
        for repeat in range(SHAPE_REPEATS):
            try:
                all_rows.append(run_one(
                    gpu, 'shape', 'cls_full_attention', repeat,
                    b_micro, concurrency, shape_env, False,
                ))
            except Exception as exc:
                errors.append({'gpu': gpu, 'phase': 'shape', 'label': f'b{b_micro}c{concurrency}', 'error': str(exc)})
                print('T4_SHAPE_ERROR', errors[-1], flush=True)

best_shape_by_gpu = {}
for gpu in BENCH_GPUS:
    candidates = []
    for b_micro, concurrency in BENCH_SHAPES:
        samples = [
            row['candidates_per_sec'] for row in all_rows
            if row['gpu'] == gpu and row['phase'] == 'shape'
            and row['b_micro'] == b_micro and row['concurrency'] == concurrency
        ]
        if samples:
            candidates.append({
                'b_micro': b_micro,
                'concurrency': concurrency,
                'median_candidates_per_sec': statistics.median(samples),
                'samples': samples,
            })
    if not candidates:
        raise RuntimeError(f'no successful shape rows for GPU {gpu}')
    best_shape_by_gpu[gpu] = max(candidates, key=lambda item: item['median_candidates_per_sec'])
    print('T4_BEST_SHAPE', gpu, best_shape_by_gpu[gpu], flush=True)

for gpu in BENCH_GPUS:
    b_micro = best_shape_by_gpu[gpu]['b_micro']
    concurrency = best_shape_by_gpu[gpu]['concurrency']
    for label, policy_env in POLICY_CONFIGS:
        for repeat in range(POLICY_REPEATS):
            try:
                all_rows.append(run_one(
                    gpu, 'policy', label, repeat,
                    b_micro, concurrency, policy_env, True,
                ))
            except Exception as exc:
                error = {'gpu': gpu, 'phase': 'policy', 'label': label, 'repeat': repeat, 'error': str(exc)}
                errors.append(error)
                print('T4_POLICY_ERROR', error, flush=True)
                break

policy_summary = []
for gpu in BENCH_GPUS:
    reference_rows = [
        row for row in all_rows
        if row['gpu'] == gpu and row['phase'] == 'policy' and row['label'] == 'full_final'
    ]
    baseline_rows = [
        row for row in all_rows
        if row['gpu'] == gpu and row['phase'] == 'policy' and row['label'] == 'cls_full_attention'
    ]
    if len(reference_rows) != POLICY_REPEATS or len(baseline_rows) != POLICY_REPEATS:
        raise RuntimeError(f'GPU {gpu} is missing required full_final or cls_full_attention repeats')
    reference_sha = reference_rows[0]['dump_sha256']
    baseline_median = statistics.median(row['candidates_per_sec'] for row in baseline_rows)
    if any(row['dump_sha256'] != reference_sha for row in reference_rows + baseline_rows):
        raise RuntimeError(f'GPU {gpu} required full-final/CLS-only exactness or determinism failed')
    for label, _ in POLICY_CONFIGS:
        rows = [
            row for row in all_rows
            if row['gpu'] == gpu and row['phase'] == 'policy' and row['label'] == label
        ]
        if not rows:
            continue
        exact = len(rows) == POLICY_REPEATS and all(row['dump_sha256'] == reference_sha for row in rows)
        median_cps = statistics.median(row['candidates_per_sec'] for row in rows)
        item = {
            'gpu': gpu,
            'label': label,
            'repeats': len(rows),
            'exact': exact,
            'reference_sha256': reference_sha,
            'median_candidates_per_sec': median_cps,
            'speedup_vs_cls_full': median_cps / baseline_median,
            'min_candidates_per_sec': min(row['candidates_per_sec'] for row in rows),
            'max_candidates_per_sec': max(row['candidates_per_sec'] for row in rows),
        }
        policy_summary.append(item)
        print('T4_POLICY_RESULT', item, flush=True)

csv_path = Path('/kaggle/working/stream1_transformer_t4_rows.csv')
fieldnames = [
    'gpu', 'phase', 'label', 'repeat', 'b_micro', 'concurrency',
    'rows_per_launch_group', 'ms_per_launch_group', 'parents_per_sec',
    'candidates_per_sec', 'checksum', 'score_key_digest', 'scratch_bytes',
    'dump_path', 'dump_sha256',
]
with csv_path.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

summary = {
    'checked_out_commit': commit,
    'github_ref': GITHUB_REF,
    'best_shape_by_gpu': best_shape_by_gpu,
    'policy_summary': policy_summary,
    'errors': errors,
}
summary_path = Path('/kaggle/working/stream1_transformer_t4_summary.json')
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

md_path = Path('/kaggle/working/stream1_transformer_t4_summary.md')
with md_path.open('w', encoding='utf-8') as fh:
    fh.write('# Stream1 transformer 2xT4 hardware sweep\n\n')
    fh.write(f'- commit: `{commit}`\n')
    fh.write(f'- ref: `{GITHUB_REF}`\n\n')
    fh.write('## Best shapes\n\n')
    for gpu, item in sorted(best_shape_by_gpu.items()):
        fh.write(f'- GPU{gpu}: b_micro={item["b_micro"]}, concurrency={item["concurrency"]}, median={item["median_candidates_per_sec"]:.1f} candidates/s\n')
    fh.write('\n## Policy results\n\n')
    fh.write('| GPU | policy | repeats | exact | median candidates/s | vs cls_full |\n')
    fh.write('|---:|---|---:|---:|---:|---:|\n')
    for item in policy_summary:
        fh.write(f'| {item["gpu"]} | {item["label"]} | {item["repeats"]} | {int(item["exact"])} | {item["median_candidates_per_sec"]:.1f} | {item["speedup_vs_cls_full"]:.4f}x |\n')
    if errors:
        fh.write('\n## Errors\n\n')
        for error in errors:
            fh.write(f'- `{error}`\n')

print('STREAM1_TRANSFORMER_T4_ROWS=', csv_path, flush=True)
print('STREAM1_TRANSFORMER_T4_SUMMARY=', summary_path, flush=True)
print('STREAM1_TRANSFORMER_T4_REPORT=', md_path, flush=True)
